# 01 — Dataset Consolidation & Exploratory Data Analysis

Merge 5 Excel files → clean → deduplicate → stratified split → comprehensive EDA.

**Instructions**: Upload your 5 Excel files to `data/raw/` before running.

In [ ]:
!pip install -q pandas openpyxl scikit-learn matplotlib seaborn

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if os.path.basename(os.getcwd()) == 'notebooks' else 'src')

from consolidate import consolidate

consolidate(
    raw_dir='data/raw',
    out_dir='data/processed',
    test_size=0.15,
    val_size=0.15,
    seed=42,
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load splits
full  = pd.read_csv('data/processed/full_clean.csv')
train = pd.read_csv('data/processed/train.csv')
val   = pd.read_csv('data/processed/val.csv')
test  = pd.read_csv('data/processed/test.csv')

print(f'Total clean samples: {len(full)}')
print(f'Train: {len(train)},  Val: {len(val)},  Test: {len(test)}')
print(f'Columns: {list(full.columns)}')
print(f'\nPrimary classes: {sorted(full["primary_label"].unique())}')
print(f'\nMissing values:\n{full.isnull().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
palette = sns.color_palette('Set2', n_colors=5)

for ax, (name, df) in zip(axes, [('Train', train), ('Val', val), ('Test', test)]):
    counts = df['primary_label'].value_counts().sort_index()
    bars = counts.plot.bar(ax=ax, color=palette, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name} (n={len(df)})', fontweight='bold')
    ax.set_ylabel('Count' if name == 'Train' else '')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
    for bar, v in zip(bars.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=9)

plt.suptitle('Label Distribution per Split', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/01_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


pct = full['primary_label'].value_counts(normalize=True).sort_index() * 100
print('Class percentages (full dataset):')
for cls, p in pct.items():
    print(f'  {cls:20s}: {p:5.1f}%')

In [ ]:
full['code_len'] = full['code'].str.len()
full['code_lines'] = full['code'].str.count('\n') + 1
full['code_tokens'] = full['code'].str.split().apply(len)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (col, label) in zip(axes, [
    ('code_len', 'Characters'), ('code_lines', 'Lines'), ('code_tokens', 'Tokens (whitespace)')
]):
    full.groupby('primary_label')[col].plot.hist(
        ax=ax, bins=40, alpha=0.6, legend=True, edgecolor='black', linewidth=0.3
    )
    ax.set_xlabel(label)
    ax.set_title(f'Code Length ({label})')

plt.tight_layout()
plt.savefig('results/01_code_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


print('\nCode length stats per class (characters):')
print(full.groupby('primary_label')['code_len'].describe().round(0).to_string())
print('\nCode length stats per class (lines):')
print(full.groupby('primary_label')['code_lines'].describe().round(0).to_string())

In [ ]:
has_sub = full[full['sublabel'].notna()]
print(f'{len(has_sub)} / {len(full)} samples ({100*len(has_sub)/len(full):.1f}%) have sublabels\n')

sub_counts = has_sub.groupby(['primary_label', 'sublabel']).size().reset_index(name='count')
sub_counts = sub_counts.sort_values(['primary_label', 'count'], ascending=[True, False])
print(sub_counts.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, cls in enumerate(sorted(has_sub['primary_label'].unique())):
    ax = axes[i]
    cls_sub = has_sub[has_sub['primary_label'] == cls]['sublabel'].value_counts().head(10)
    cls_sub.plot.barh(ax=ax, color=palette[i], edgecolor='black', linewidth=0.3)
    ax.set_title(f'{cls} sublabels', fontweight='bold')
    ax.invert_yaxis()
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig('results/01_sublabel_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Box-plots: code length per class ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=full, x='primary_label', y='code_len', ax=axes[0],
            palette=palette, showfliers=False)
axes[0].set_title('Character Length per Class (no outliers)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=full, x='primary_label', y='code_lines', ax=axes[1],
            palette=palette, showfliers=False)
axes[1].set_title('Line Count per Class (no outliers)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('results/01_code_length_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Move keyword frequency per class ---
move_keywords = [
    'module', 'public', 'fun', 'struct', 'use', 'let', 'mut', 'entry',
    'has', 'copy', 'drop', 'store', 'key', 'abort', 'assert!', 'transfer',
    'object', 'sui', 'vector', 'option', 'tx_context', 'borrow', 'return',
    'if', 'else', 'while', 'loop', 'move', 'acquires', 'native',
]

keyword_data = []
for _, row in full.iterrows():
    code_lower = row['code'].lower()
    counts = {kw: code_lower.count(kw) for kw in move_keywords}
    counts['primary_label'] = row['primary_label']
    keyword_data.append(counts)

kw_df = pd.DataFrame(keyword_data)
kw_means = kw_df.groupby('primary_label')[move_keywords].mean()

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(kw_means.T, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean count'})
ax.set_title('Move Keyword Frequency per Class (mean)', fontweight='bold')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('results/01_keyword_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Sample code snippets per class ---
for cls in sorted(full['primary_label'].unique()):
    subset = full[full['primary_label'] == cls]
    sample = subset.sample(1, random_state=42).iloc[0]
    print(f'\n{"="*60}')
    print(f'CLASS: {cls}  |  Label: {sample["label"]}')
    print(f'Code length: {len(sample["code"])} chars, {sample["code"].count(chr(10))+1} lines')
    print(f'{"="*60}')
    # Show first 20 lines
    lines = sample['code'].split('\n')[:20]
    print('\n'.join(lines))
    if len(sample['code'].split('\n')) > 20:
        print(f'  ... ({len(sample["code"].split(chr(10))) - 20} more lines)')
    print(f'\nOutput/Fix: {sample["output"][:200]}')

In [ ]:
# --- Source file contribution ---
source_counts = full.groupby(['source_file', 'primary_label']).size().unstack(fill_value=0)
print('Samples per source file × class:')
print(source_counts.to_string())
print(f'\nTotal per file: {full["source_file"].value_counts().to_dict()}')

In [ ]:
# --- EDA Summary ---
print('='*60)
print('  DATASET SUMMARY')
print('='*60)
print(f'  Total clean samples   : {len(full)}')
print(f'  Primary classes       : {len(full["primary_label"].unique())}')
print(f'  Samples with sublabels: {full["sublabel"].notna().sum()} ({100*full["sublabel"].notna().mean():.1f}%)')
print(f'  Unique sublabels      : {full["sublabel"].nunique()}')
print(f'  Median code length    : {full["code_len"].median():.0f} chars / {full["code_lines"].median():.0f} lines')
print(f'  Train / Val / Test    : {len(train)} / {len(val)} / {len(test)}')
print()
print('  Class imbalance ratio (largest / smallest):', end=' ')
vc = full['primary_label'].value_counts()
print(f'{vc.max() / vc.min():.1f}x  ({vc.idxmax()} vs {vc.idxmin()})')
print()
print('  Observations:')
print('  - Dominant class "Perfect" has ~47% of samples')
print('  - "SyntaxError" is underrepresented (~4%) — consider class weights')
print(f'  - Code lengths vary widely (min={full["code_len"].min()}, max={full["code_len"].max()} chars)')
print('='*60)